# 한일합섬 ERP DB 검색기 (노트북)

에이전트 없이 사람이 직접 DB 를 살펴보는 노트북입니다. 각 절의 셀을 실행하면 **입력창**이 뜹니다 (VS Code 는 창 맨 위 가운데에 표시됩니다). 값을 넣고 Enter, 비워 두고 Enter 를 치면 `[ ]` 안의 기본값으로 실행됩니다.

| 절 | 하는 일 | 입력 |
| --- | --- | --- |
| 0 | 접속 · 권한 확인 | — |
| 1 | 테이블 찾기 | 이름 일부 (비우면 데이터 있는 전체 목록) |
| 2 | 컬럼 보기 | 테이블명, 컬럼 검색어 |
| 3 | 데이터 조회 | 테이블명 · 컬럼 · 조건 · 정렬 · 건수 |
| 4 | 값으로 찾기 | 테이블명 · 컬럼 · 찾을 값 |
| 5 | 값별 건수 (분포) | 테이블명 · 컬럼 · 조건 |
| 6 | 자유 SQL | 셀 안의 SQL 을 직접 수정 |
| 7 | 코드값 뜻 찾기 | 코드 필드 한글명 |
| 8 | 결과 저장 | 파일명 |
| 9 | 목적으로 테이블 찾기 | 업무 용어 |
| 10 | 테이블 근거 팩 | 테이블명 |

- **앞 셀에서 입력한 값이 뒷 셀의 기본값으로 이어집니다.** 테이블명은 2 · 3 · 4 · 5절이 공유하고, 컬럼명은 테이블별로, 최대 건수와 코드 필드명도 마지막 값을 기억합니다. 입력창 `[ ]` 안이 이번에 적용될 값이니, 그대로 쓰려면 비우고 Enter, 바꾸려면 새로 입력합니다.
- 조건(WHERE) · 정렬 · 컬럼 목록은 기억하지 않습니다. 비우면 '조건 없음'이라 이전 조건이 몰래 다시 걸리지 않습니다.
- 조회 결과는 항상 변수 `df` 에 남으므로, 8절에서 마지막 결과를 저장할 수 있습니다.
- 일자 컬럼(`DT_*`)은 `'YYYYMMDD'` **문자열**입니다. 조건에 `DT_SO >= '20260901'` 처럼 적습니다. 회사코드는 환경별로 확인하고 예시 값을 바꾸세요.
- 운영 서버라 보호 장치가 켜져 있습니다: 결과 10,000행 · 쿼리 60초 · 큰 테이블에 조건 없는 조회 차단. 막히면 조건을 좁히는 것이 맞습니다.

**준비**: 커널을 이 저장소의 `.venv` 로 선택하고(없으면 `uv venv && uv pip install -e ".[notebook]"`), 접속정보 `.env` 를 이 노트북과 같은 폴더에 둡니다.
**실행 방법**: 0절을 먼저 실행하고 필요한 절만 Shift+Enter로 실행하세요. `모두 실행`은 여러 입력창과 조회를 연속 실행하므로 권장하지 않습니다. 쿼리 실패·취소 시 `df`는 비워집니다.


## 0. 접속 · 권한 확인

In [ ]:
import hhhs_db_manager as db  # TLS 설정을 위해 DB 드라이버보다 먼저 import
import pandas as pd
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


def ask(prompt, default="", key=None):
    """입력창. 비우면 [ ] 안의 값이 쓰인다.

    key 를 주면 앞 셀에서 마지막으로 쓴 값이 기본값이 된다 (테이블명 등).
    기억한 값은 ask.memory 에 남는다. key 가 없는 항목(조건 · 정렬 · 컬럼 목록)은
    비우면 '조건 없음' 그대로라 이전 조건이 몰래 다시 걸리지 않는다.
    """
    memory = ask.__dict__.setdefault("memory", {})
    fallback = memory.get(key, default) if key else default
    value = input(f"{prompt}" + (f" [{fallback}]" if fallback else "") + ": ").strip() or fallback
    if key and value:
        memory[key] = value
    return value


def show(result, all_rows=False):
    """행수를 찍고 표를 보여 준 뒤 결과를 돌려준다 (df 변수에 담긴다)."""
    note = "  ⚠️ 상한에서 잘림 — 조건을 좁히세요" if result.attrs.get("truncated") else ""
    print(f"{len(result):,}행{note}")
    with pd.option_context("display.max_rows", None if all_rows else 200):
        display(result)
    return result


def identifier(value):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", value):
        raise ValueError("테이블·컬럼명에는 영문, 숫자, 밑줄만 사용하세요.")
    return f"[{value}]"


def full_name(table):
    parts = table.strip().split(".")
    if len(parts) == 1:
        parts = [db.SCHEMA] + parts
    if len(parts) != 2:
        raise ValueError("테이블명 또는 스키마.테이블명으로 입력하세요.")
    return ".".join(identifier(p) for p in parts)


def ask_limit():
    while True:
        raw = ask("최대 건수 (1~10000)", "100", key="limit")
        if raw.isdecimal() and 1 <= int(raw) <= 10000:
            return int(raw)
        print("1~10000 사이 정수를 입력하세요.")


# 저장 위치는 프로젝트 루트. 저장소가 DB조회도구 이름이면 그 상위 폴더입니다.
ROOT = Path.cwd().resolve()
if ROOT.name == "DB조회도구":
    ROOT = ROOT.parent
print("결과 저장 폴더:", ROOT)
info = db.check(verbose=False)
print("접속 확인 완료 · 쓰기 권한 점검 대상 테이블 수:", info["writable_tables_in_schema"])
if info["writable_tables_in_schema"]:
    raise RuntimeError("쓰기 권한이 발견되었습니다. 담당자 확인 후 사용하세요.")


## 1. 테이블 찾기

이름 일부로 찾습니다. 비우면 **데이터가 있는 전체 테이블**을 행수 많은 순으로 전부 보여 줍니다 (빈 테이블 3,600여 개는 제외).

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
keyword = ask("테이블 이름에 들어갈 문자열 (예: SO, QTIO, PRQ, ITEM) — 비우면 데이터 있는 전체")

df = show(db.get_list_tables(keyword or None, min_rows=1).sort_values("rows", ascending=False), all_rows=True)

## 2. 컬럼 보기

`name_kr` 은 ERP 사전의 한글명, `pk` 는 기본키 순서. 컬럼 검색어는 컬럼명(예 `DT_`)이나 한글명(예 `일자`) 어느 쪽이든 됩니다.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
table = ask("테이블명 (스키마 생략 시 NEOE)", "SA_SOH", key="table")
col_kw = ask("컬럼 검색어 — 컬럼명 또는 한글명 (비우면 전체)")

cols = db.get_columns(table)
if col_kw:
    cols = cols[cols["column"].str.contains(col_kw, case=False, regex=False)
                | cols["name_kr"].fillna("").str.contains(col_kw, regex=False)]
df = show(cols, all_rows=True)

## 3. 데이터 조회

`SELECT TOP 건수 컬럼 FROM 테이블 WHERE 조건 ORDER BY 정렬` 을 입력값으로 만듭니다. 조건은 SQL 그대로 적습니다 — 예: `DT_SO >= '20260901' AND CD_COMPANY = '1000'`, `NM_ITEM LIKE '%POLY%'`.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
table = ask("테이블명", "SA_SOH", key="table")
columns = ask("컬럼 (쉼표 구분, 비우면 전체)")
where = ask("조건 WHERE (비우면 없음)")
order = ask("정렬 ORDER BY (예: DT_SO DESC, 비우면 없음)")
limit = ask_limit()

df = show(db.get_table(
    table, limit,
    columns=[c.strip() for c in columns.split(",") if c.strip()] or None,
    where=where or None,
    order_by=order or None,
))

## 4. 값으로 찾기

어떤 컬럼에 특정 값(의 일부)이 들어간 행을 찾습니다. 품목명으로 품목코드 찾기, 거래처명으로 거래처코드 찾기에 씁니다.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
table = ask("테이블명", "MA_ITEM", key="table")
column = ask("찾을 컬럼", "NM_ITEM" if table == "MA_ITEM" else "", key=f"컬럼@{table}")
if not column:
    raise ValueError(f"{table} 의 컬럼명을 입력하세요. 2절에서 컬럼 목록을 확인할 수 있습니다.")
value = ask("찾을 값 (일부만 적어도 됨)", "POLY")
limit = ask_limit()

df = show(db.get_table(table, limit, where=f"{identifier(column)} LIKE :v", v=f"%{value}%"))

## 5. 값별 건수 (분포)

컬럼에 어떤 값이 몇 건씩 있는지 봅니다. 상태 · 구분 코드가 실제로 어떻게 쓰이는지 볼 때 유용합니다. 뜻은 7절에서 찾습니다.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
table = ask("테이블명", "SA_SOH", key="table")
column = ask("집계할 컬럼", "STA_SO" if table == "SA_SOH" else "", key=f"컬럼@{table}")
if not column:
    raise ValueError(f"{table} 의 컬럼명을 입력하세요. 2절에서 컬럼 목록을 확인할 수 있습니다.")
where = ask("조건 WHERE (비우면 없음)")

sql = (f"SELECT TOP 50 {identifier(column)} AS 값, COUNT(*) AS 건수 FROM {full_name(table)}"
       + (f" WHERE {where}" if where else "")
       + f" GROUP BY {identifier(column)} ORDER BY 건수 DESC")
df = show(db.query(sql))

## 6. 자유 SQL

셀 안의 SQL 을 직접 고쳐 실행합니다. 규칙: 테이블 앞에 `NEOE.`, 탐색은 `TOP N`, 값은 `:이름` 으로 두고 `db.query(sql, 이름=값)` 으로 넘기면 따옴표 걱정이 없습니다.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
sql = """
SELECT TOP 20 H.NO_SO, H.DT_SO, H.CD_PARTNER, P.LN_PARTNER AS 거래처명, H.STA_SO
FROM NEOE.SA_SOH H
LEFT JOIN NEOE.MA_PARTNER P ON P.CD_COMPANY = H.CD_COMPANY AND P.CD_PARTNER = H.CD_PARTNER
WHERE H.CD_COMPANY = :co AND H.DT_SO >= :d
ORDER BY H.DT_SO DESC
"""

df = show(db.query(sql, co="1000", d="20260901"))

## 7. 코드값 뜻 찾기

`STA_SO = 'R'` 처럼 코드로 저장된 값의 뜻은 ERP 코드표(`MA_CODE` · `MA_CODEDTL`)에 있습니다. 필드의 한글명 일부로 찾습니다.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
field = ask("코드 필드 한글명 (예: 수주상태, 작업상태, 수불구분, 품목)", "수주상태", key="field")

df = show(db.query("""
    SELECT c.CD_FIELD, c.NM_FIELD AS 필드, d.CD_SYSDEF AS 값, d.NM_SYSDEF AS 뜻
    FROM NEOE.MA_CODEDTL d
    JOIN NEOE.MA_CODE c ON c.CD_FIELD = d.CD_FIELD AND c.CD_COMPANY = d.CD_COMPANY
    WHERE d.CD_COMPANY = 'MASTER' AND c.NM_FIELD LIKE :f AND d.USE_YN = 'Y'
    ORDER BY c.CD_FIELD, d.CD_SYSDEF
""", f=f"%{field}%"), all_rows=True)

## 8. 결과 저장

마지막 성공 결과(`df`)를 프로젝트 루트에 저장합니다. 파일명만 입력하세요. 기존 파일은 덮어쓰지 않으며, 같은 이름이 있으면 다른 이름을 사용하세요. 조회 결과는 사내 자료이므로 외부로 보내지 마세요. `.xlsx` 저장 패키지는 notebook 추가 의존성에 포함됩니다.


In [ ]:
if globals().get("df") is None:
    print("저장할 결과가 없습니다. 1~7절 중 하나를 성공적으로 실행하세요.")
else:
    filename = ask("저장 파일명 (.csv 또는 .xlsx)", "결과.csv")
    path = ROOT / filename
    if Path(filename).name != filename or path.suffix.lower() not in {".csv", ".xlsx"}:
        raise ValueError("폴더 경로 없이 .csv 또는 .xlsx 파일명만 입력하세요.")
    if path.is_symlink():
        raise ValueError("심볼릭 링크에는 저장하지 않습니다.")
    if df.attrs.get("truncated"):
        print("주의: 전체 결과가 아닌 상한에서 잘린 결과입니다.")
    # 기존 자료(보안 태그 포함)를 덮어쓰지 않습니다.
    with path.open("xb") as target:
        if path.suffix.lower() == ".xlsx":
            df.to_excel(target, index=False, engine="openpyxl")
        else:
            df.to_csv(target, index=False, encoding="utf-8-sig")
    print(f"저장: {path} ({len(df):,}행)")


## 9. 목적으로 테이블 찾기

업무 용어로 후보 테이블을 찾습니다. 사전(`테이블사전.md`)의 한글명·설명, 테이블 이름, **컬럼 한글명**을 대조해 점수를 매기고 `근거` 열에 무엇이 맞았는지 보여 줍니다. 사전이 없으면 터미널에서 `hhhs-db dict init` 을 한 번 실행해 두세요.

In [ ]:
df = None  # 실패·취소 시 이전 조회 결과를 실수로 저장하지 않도록 초기화
purpose = ask("찾는 목적 · 키워드 (예: 수주 납기, 롤 폭 길이, 거래처 여신)", "수주 납기")

df = show(db.find_tables(purpose, limit=20), all_rows=True)

## 10. 테이블 근거 팩 (이 테이블은 무엇인가)

ERP 사전 · 컬럼 한글명 · MS_Description · PK · 같은 이름줄기의 이웃 · 이름이 닮은 ERP 화면 · 사전 현재 항목을 한 번에 봅니다. 뜻을 판단해 사전에 적을 때는 터미널에서 `hhhs-db dict add 테이블 --name-kr … --desc … --basis …`.

In [ ]:
table = ask("근거 팩을 볼 테이블명", "SA_SOH", key="table")

print(db.describe_text(db.describe(table)))

## 막히면

| 메시지 | 조치 |
| --- | --- |
| `⚠️ 상한에서 잘림` / `UserWarning: 결과가 10,000행에서 잘렸습니다` | 조건을 좁히거나 6절에서 `db.query(sql, max_rows=50000)` |
| `QueryRejected: … 100만 행 이상인데 TOP · WHERE · 집계가 없습니다` | 조건(WHERE)에 기간을 넣기 |
| `QueryTimeout` | 범위를 줄이기. 제한시간 변경은 실행 전 환경변수로 설정하고 커널 재시작 (README 참고) |
| `DBError: 테이블이 없습니다` / `컬럼이 없습니다` | 1절 · 2절로 이름 확인 (스키마 이름 `NEOE` 는 테이블명이 아닙니다) |
| `DBError: TLS 협상 실패` | 커널 재시작 후 0절부터 다시 |
| 그 밖의 오류 | README 「오류가 나면」 표 |

노트북을 커밋할 때는 「모든 출력 지우기」를 먼저 하세요. 실행 결과가 파일에 남습니다.